# Split-RAG experiments
This notebook implements the Split-RAG architecture, a specialized retrieval strategy designed to overcome the limitations identified during the baseline experiments. While the baseline Naive RAG treated legislation and jurisprudence as a single unstructured corpus, Split-RAG introduces a functional decoupling of these data sources.

**Baseline RAGAs results**: 
* faithfulness: 0.7205 
* factual correctness: 0.3340
* context precision: 0.8750
* context recall: 0.3417

## Imports

In [21]:
import os
import time
import pandas as pd
import json
from datasets import Dataset
from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import AzureOpenAIEmbeddings
from langchain_chroma import Chroma
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AsyncAzureOpenAI
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.storage import LocalFileStore, create_kv_docstore



from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
from ragas.llms import llm_factory
import nest_asyncio
from ragas import aevaluate 
from ragas import RunConfig



C:\Users\verkad004\AppData\Local\Temp\ipykernel_1896\1394475783.py:21: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
C:\Users\verkad004\AppData\Local\Temp\ipykernel_1896\1394475783.py:21: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
C:\Users\verkad004\AppData\Local\Temp\ipykernel_1896\1394475783.py:21: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead

## Environment variables

In [4]:
env_path = os.path.join("..", ".env")
load_dotenv(dotenv_path=env_path)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
embedding_deployment = os.getenv("AZURE_EMBEDDING_DEPLOYMENT")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT")

# Initialize Azure AD token provider
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

## Data Loading
In contrast to the baseline experiments, the Split-RAG architecture employs a decoupled data strategy. Rather than treating legislation and jurisprudence as a unified corpus, this approach maintains them as two distinct, specialized knowledge bases.

In [5]:
legislation_path = '../data/legislation_omgevingswet_cleaned.jsonl'
jurisprudence_path = '../data/jurisprudence_omgevingswet_cleaned.jsonl'

common_cols = ['id', 'title', 'text', 'word_count', 'source_type']
df_legislation = pd.read_json(legislation_path, lines=True)
df_jurisprudence = pd.read_json(jurisprudence_path, lines=True)

df_leg_sub = df_legislation[common_cols]
df_jur_sub = df_jurisprudence[common_cols]

print(f"Legislation loaded: {len(df_legislation)} rows")
print(f"Jurisprudence loaded: {len(df_jurisprudence)} rows")

Legislation loaded: 725 rows
Jurisprudence loaded: 3404 rows


In [6]:
# Legislation document creation
docs_legislation = [
    Document(
        page_content=str(row['text']),
        metadata={
            "id": row['id'],
            "title": row['title'],
            "source_type": row['source_type'],
            "word_count": row['word_count']
        }
    ) for _, row in df_leg_sub.iterrows()
]

# jurisprudence document creation
docs_jurisprudence = [
    Document(
        page_content=str(row['text']),
        metadata={
            "id": row['id'],
            "title": row['title'],
            "source_type": row['source_type'],
            "word_count": row['word_count']
        }
    ) for _, row in df_jur_sub.iterrows()
]

print(f"Created {len(docs_legislation)} legislation documents.")
print(f"Created {len(docs_jurisprudence)} jurisprudence documents.")

Created 725 legislation documents.
Created 3404 jurisprudence documents.


# Retrieving documents per dataset

In [7]:
embeddings = AzureOpenAIEmbeddings(
    azure_deployment=embedding_deployment, 
    azure_endpoint=endpoint,              
    openai_api_version=api_version,       
    azure_ad_token_provider=token_provider
)

## Legislation
For legislation, we are implementing a **Parent Document Retrieval (PDR)** system. Instead of standard chunking, this system employs a hierarchical approach.

**Rationale:**
Laws are inherently hierarchical and logically structured. Recent research (Lim et al., 2025; Guo et al., 2025) demonstrates that multi-granular context is superior to fixed chunk sizes. By indexing small child chunks (400 tokens), we maximise search precision. However, to prevent the AI from losing the legal context, the full parent” article is served to the model as context upon a match. This chunk-to-context reconstruction directly addresses the shortcomings of naive RAG systems when dealing with complex legislation (Lim et al., 2025).


In [ ]:
# Splitter for search chunks (childs)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, 
    chunk_overlap=50
)

# Define paths
vdb_base_path = "../data/vector_stores"
leg_child_path = os.path.join(vdb_base_path, "split_legislation_child")
leg_parent_path = os.path.join(vdb_base_path, "split_legislation_parent_store")

# Parent storage for disk
fs = LocalFileStore(leg_parent_path)
parent_docstore = create_kv_docstore(fs) 

# Child database
vdb_leg_child = Chroma(
    collection_name="leg_pdr_final",
    embedding_function=embeddings, 
    persist_directory=leg_child_path
)

# Retriever that links child chunks
# cosine similarity
retriever_leg_parent = ParentDocumentRetriever(
    vectorstore=vdb_leg_child,
    docstore=parent_docstore,
    child_splitter=child_splitter,
)

In [ ]:
# Check if disk is empty
if vdb_leg_child._collection.count() == 0:
    print("Vectordatabase empty")
    batch_size = 50
    
    for i in range(0, len(docs_legislation), batch_size):
        batch = docs_legislation[i : i + batch_size]
        # Retriever splits them here into chunks of 400 (child) and saves the entire doc (parent)
        retriever_leg_parent.add_documents(batch, ids=None)
        
        print(f"Progress: {min(i + batch_size, len(docs_legislation))}/{len(docs_legislation)} articles done")
        time.sleep(1)
    print("Database is ready")
else:
    print("Database is ready")

Vectordatabase is leeg. Start met vullen...
Voortgang: 50/725 artikelen verwerkt
Voortgang: 100/725 artikelen verwerkt
Voortgang: 150/725 artikelen verwerkt
Voortgang: 200/725 artikelen verwerkt
Voortgang: 250/725 artikelen verwerkt
Voortgang: 300/725 artikelen verwerkt
Voortgang: 350/725 artikelen verwerkt
Voortgang: 400/725 artikelen verwerkt
Voortgang: 450/725 artikelen verwerkt
Voortgang: 500/725 artikelen verwerkt
Voortgang: 550/725 artikelen verwerkt
Voortgang: 600/725 artikelen verwerkt
Voortgang: 650/725 artikelen verwerkt
Voortgang: 700/725 artikelen verwerkt
Voortgang: 725/725 artikelen verwerkt
Klaar! Alles staat nu op de schijf.


## Jurisprudence
For the jurisprudence, we use Structure-Aware Chunking.

**Rationale:**
Jurisprudence is narrative and story-driven in nature, meaning that relevant information is often scattered across lengthy texts. In line with the findings in *LegalBench-RAG* (Pipitone & Alami, 2024), standard retrieval is often insufficient for this type of document. We use a splitter that preserves the logical integrity of the judgment (splitting by ECLI and paragraphs). 

In [11]:
structure_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    length_function=len,
    separators=[
        "\nECLI:",          # Prioriteit voor de start van een nieuwe uitspraak
        "\n\n",            # Paragrafen
        "\n",              # Regels
        ". ",              # Zinnen
        " "                # Woorden
    ]
)

# Pas de splitter toe op je jurisprudentie documenten
chunks_juris_structure = structure_splitter.split_documents(docs_jurisprudence)

print(f"Jurisprudence: Structure-Aware Results")
print(f"Totaal aantal chunks: {len(chunks_juris_structure)}")

# Check de eerste chunk
if chunks_juris_structure:
    print(f"\nSample Metadata: {chunks_juris_structure[0].metadata}")
    print(f"Sample Content: {chunks_juris_structure[0].page_content[:300]}...")

Jurisprudence: Structure-Aware Results
Totaal aantal chunks: 51321

Sample Metadata: {'id': 'ECLI:NL:RBGEL:2024:26', 'title': 'ECLI:NL:RBGEL:2024:26, Rechtbank Gelderland, 05-01-2024, AWB-22_5249 en 22_5252', 'source_type': 'jurisprudence', 'word_count': 2403}
Sample Content: Weigering handhavingsverzoeken m.b.t. geitenhouderij. Intern salderen. Beroep gegrond vanwege een motiveringsgebrek. De rechtsgevolgen worden door de rechtbank in stand gelaten omdat in het verweerschrift afdoende is onderbouwd dat er geen sprake is van een overtreding van de Wnb.


    

Zittingspl...


In [14]:
def create_vdb_in_batches(chunks, path, name, embeddings, batch_size=50):
    print(f"Checking existing progress for: {name}")
    
    # 1. Open of maak de database aan
    vector_db = Chroma(
        persist_directory=path,
        embedding_function=embeddings
    )
    
    # 2. Kijk wat er al in zit (op basis van metadata 'id' of 'source')
    # We halen de unieke bronnen op die al verwerkt zijn
    existing_count = vector_db._collection.count()
    
    if existing_count > 0:
        print(f"Hervatten: Er staan al {existing_count} chunks in de database.")
        # We skippen de chunks die al aanwezig zijn
        # Let op: dit werkt het best als de volgorde van 'chunks' altijd hetzelfde is
        remaining_chunks = chunks[existing_count:]
    else:
        print("Geen bestaande data gevonden. We starten vanaf het begin.")
        remaining_chunks = chunks

    if not remaining_chunks:
        print(f"Alle chunks voor {name} zijn al verwerkt!\n")
        return vector_db

    # 3. Voeg de resterende chunks toe in batches
    print(f"Nog {len(remaining_chunks)} chunks toe te voegen...")
    
    for i in range(0, len(remaining_chunks), batch_size):
        batch = remaining_chunks[i : i + batch_size]
        vector_db.add_documents(documents=batch)
        
        current_total = existing_count + i + len(batch)
        print(f"Progress for {name}: {current_total}/{len(chunks)} chunks totaal...")
        
        # Voorkom rate limits en geef de schijf tijd om te schrijven
        time.sleep(1) 
        
    print(f"{name} Vector Store succesvol bijgewerkt bij {path}\n")
    return vector_db

# Pad voor de jurisprudentie database
juris_vdb_path = os.path.join(vdb_base_path, "split_jurisprudence_structure")

# Gebruik je vertrouwde functie om de vdb aan te maken
vdb_juris_split = create_vdb_in_batches(
    chunks=chunks_juris_structure, 
    path=juris_vdb_path, 
    name="Jurisprudence-Structure",
    embeddings=embeddings,
    batch_size=50
)

Checking existing progress for: Jurisprudence-Structure
Hervatten: Er staan al 27400 chunks in de database.
Nog 23921 chunks toe te voegen...
Progress for Jurisprudence-Structure: 27450/51321 chunks totaal...
Progress for Jurisprudence-Structure: 27500/51321 chunks totaal...
Progress for Jurisprudence-Structure: 27550/51321 chunks totaal...
Progress for Jurisprudence-Structure: 27600/51321 chunks totaal...
Progress for Jurisprudence-Structure: 27650/51321 chunks totaal...
Progress for Jurisprudence-Structure: 27700/51321 chunks totaal...
Progress for Jurisprudence-Structure: 27750/51321 chunks totaal...
Progress for Jurisprudence-Structure: 27800/51321 chunks totaal...
Progress for Jurisprudence-Structure: 27850/51321 chunks totaal...
Progress for Jurisprudence-Structure: 27900/51321 chunks totaal...
Progress for Jurisprudence-Structure: 27950/51321 chunks totaal...
Progress for Jurisprudence-Structure: 28000/51321 chunks totaal...
Progress for Jurisprudence-Structure: 28050/51321 chun

In [15]:
# Load database
vdb_juris_split = Chroma(
    persist_directory=os.path.join(vdb_base_path, "split_jurisprudence_structure"),
    embedding_function=embeddings
)

print(f"Jurisprudence chunks geladen: {vdb_juris_split._collection.count()}")

Jurisprudence chunks geladen: 51321


In [16]:
retriever_juris_structure = vdb_juris_split.as_retriever(
    search_type="mmr", 
    search_kwargs={
        "k": 15,                # fetch 15 fragments
        "fetch_k": 50,          # fetch 50 and retrieve 15 most diverse
        "lambda_mult": 0.5      # balance between relevance and diversity
    }
)

## Split-RAG

In [17]:
vdb_path = "../data/vector_stores"

vdb_leg_child = Chroma(
    persist_directory=os.path.join(vdb_path, "split_legislation_child"),
    embedding_function=embeddings
)

# Load parent store (complete articles)
fs_leg = LocalFileStore(os.path.join(vdb_path, "split_legislation_parent_store"))
parent_docstore = create_kv_docstore(fs_leg)

vdb_juris_split = Chroma(
    persist_directory=os.path.join(vdb_path, "split_jurisprudence_structure"),
    embedding_function=embeddings
)

## RAGAs evaluation

### QA pairs

In [18]:
# Load JSON file
file_path = "../data/QA_pairs_evaluation.json" 

with open(file_path, 'r', encoding='utf-8') as f:
    qa_list = json.load(f)

# Convert to DataFrame
df_qa = pd.DataFrame(qa_list)
print(f"Dataset geladen: {len(df_qa)} vragen gevonden.")

Dataset geladen: 10 vragen gevonden.


### RAGAs dataset generations

In [19]:
client = AsyncAzureOpenAI(
    azure_endpoint=endpoint,
    azure_deployment=deployment,
    api_version=api_version,
    azure_ad_token_provider=token_provider,
)

In [22]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[retriever_leg_parent, retriever_juris_structure],
    weights=[0.5, 0.5] 
)

In [23]:
async def run_evaluation_loop(retriever, dataset_list, strategy_name): 
    questions = []
    all_contexts = []
    ground_truths = []
    all_answers = []


    system_prompt = (
        "Je bent een juridisch assistent voor de Gemeente Amsterdam. "
        "Beantwoord de vraag uitsluitend op basis van de verstrekte context.\n\n"
        "EISEN:\n"
        "1. Noem het specifieke wetsartikel uit de Omgevingswet.\n"
        "2. Noem het ECLI-nummer van de relevante uitspraak.\n"
        "3. Als informatie ontbreekt, geef dit dan expliciet aan."
    )


    print(f"Start retrieval and generation for: {strategy_name}...")

    for item in dataset_list:
        q = item['question']
        gt = item['ground_truth']
        
        # Retrieval
        docs = retriever.invoke(q) 
        ctx_list = [doc.page_content for doc in docs]
        context_text = "\n\n".join(ctx_list)
        
        # Generation
        resp = await client.chat.completions.create(
            model=deployment,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Context: {context_text}\n\nVraag: {q}"},
            ],
            temperature=0 
        )
        answer = resp.choices[0].message.content
        
        # Save to lists
        questions.append(q)
        all_contexts.append(ctx_list) 
        ground_truths.append(gt)
        all_answers.append(answer)

    # Dataset object for RAGAs
    ds = Dataset.from_dict({
        "question": questions,
        "answer": all_answers,
        "contexts": all_contexts,
        "ground_truth": ground_truths
    })
    
    os.makedirs("../data/results", exist_ok=True)
    ds.to_pandas().to_csv(f"../data/results/results_{strategy_name}.csv", index=False, encoding='utf-16')
    
    return ds

dataset_split = await run_evaluation_loop(ensemble_retriever, qa_list, "split-rag")

Start retrieval and generation for: split-rag...


## Results and analysis

In [24]:
evaluator_llm = llm_factory(
    model=deployment,
    client=client,
    max_tokens=4096
    )

config = RunConfig(
    timeout=240,     
    max_retries=20, 
    max_wait=60,
    max_workers=1,
    seed=42,      
)

nest_asyncio.apply()

# Initialize metrics
metrics = [
    Faithfulness(llm=evaluator_llm),
    FactualCorrectness(llm=evaluator_llm), 
    ContextPrecision(llm=evaluator_llm),
    ContextRecall(llm=evaluator_llm)
]

# evaluate
result_split= await aevaluate(
    dataset=dataset_split,
    metrics=metrics,
    run_config=config
)


print(f"results split RAG: {result_split}")


C:\Users\verkad004\AppData\Local\Temp\ipykernel_1896\425720541.py:26: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  result_split= await aevaluate(
Evaluating: 100%|██████████| 40/40 [11:22<00:00, 17.07s/it]

results split RAG: {'faithfulness': 0.8483, 'factual_correctness(mode=f1)': 0.3610, 'context_precision': 0.4594, 'context_recall': 0.6983}
